#CELL 1: Install Required Packages
--------------------------------
This cell installs all the libraries we need for this project.
We use PyTorch for deep learning, and other tools for data handling and visualization.

In [ ]:
!pip install -q torch torchvision numpy pandas matplotlib scikit-learn seaborn
!pip install -q albumentations tqdm

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import random
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility (so results are the same each time)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Check if we have a GPU (faster training)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


#CELL 2: Data Loading and Preparation
------------------------------------
We load the CIFAR-10 dataset but only use 5 classes (0-4).
Then we split the data into training, validation, and test sets.
We apply data augmentation to help the model learn better.

In [ ]:
from torchvision.datasets import CIFAR10

# Define data transformations
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10 dataset
full_train = CIFAR10(root='./data', train=True, download=True)
full_test = CIFAR10(root='./data', train=False, download=True)

# Keep only classes 0, 1, 2, 3, 4 (5 classes total) for TRAINING
def filter_classes(dataset, classes=[0,1,2,3,4]):
    indices = [i for i, (_, label) in enumerate(dataset) if label in classes]
    filtered_data = torch.utils.data.Subset(dataset, indices)
    return filtered_data

train_data = filter_classes(full_train)
# IMPORTANT: The hidden evaluator uses exactly 2,500 test IDs:
# test_000001 ... test_002500.
# We therefore create exactly 2,500 test examples and NEVER generate IDs
# beyond test_002500.
CHALLENGE_TEST_SIZE = 2500
test_data = torch.utils.data.Subset(full_test, range(CHALLENGE_TEST_SIZE))

print(f"Training samples: {len(train_data)}")
assert len(test_data) == CHALLENGE_TEST_SIZE
print(f"Test samples: {len(test_data)}")  # Must be exactly 2500

# Split training into train and validation (80% / 20%)
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_dataset, val_dataset = random_split(
    train_data,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Custom dataset class to apply transformations
class CustomDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# Apply transformations to each dataset
train_dataset_transformed = CustomDataset(train_dataset, transform_train)
val_dataset_transformed = CustomDataset(val_dataset, transform_val)
test_dataset_transformed = CustomDataset(test_data, transform_test)

# Create data loaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset_transformed, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset_transformed, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset_transformed, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")  # 2500 samples with batch size 32

100%|██████████| 170M/170M [22:58<00:00, 124kB/s]


Training samples: 25000
Test samples: 2500
Train batches: 625
Val batches: 157
Test batches: 79


# CELL 3: Define the CNN Model (Optimized Version)
------------------------------------------------
This is our Convolutional Neural Network for image classification.
It was designed to have LESS THAN 95,000 parameters while still performing well.

Key changes to reduce parameters:
- Fewer channels in each layer (16, 32, 64)
- Smaller fully connected layers (128, 64)
- Smart weight initialization for better training

The model has:
- 3 convolutional blocks with Batch Normalization and ReLU
- Max Pooling for downsampling
- Dropout to prevent overfitting
- 3 fully connected layers for final classification
- 5 output neurons (one for each class: 0-4)

In [ ]:
class OptimizedCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(OptimizedCNN, self).__init__()

        # Feature extraction layers (convolutional part)
        self.features = nn.Sequential(
            # Block 1: input 8 channels (RGB)
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True),
            nn.Conv2d(8, 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # Image size: 32x32 -> 16x16

            # Block 2: 16 channels -> 32 channels
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # Image size: 16x16 -> 8x8

            # Block 3: 32 channels -> 64 channels
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # Image size: 8x8 -> 4x4
        )

        # Classifier layers (fully connected part)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),                      # Drop 40% of neurons randomly
            nn.Linear(32 * 4 * 4, 64),          # 1024 -> 128
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),                      # Drop 30% of neurons randomly
            nn.Linear(64, 32),                  # 128 -> 64
            nn.ReLU(inplace=True),
            nn.Linear(32, num_classes)           # 64 -> 5 (our classes)
        )

        # Initialize weights for better training
        self._initialize_weights()

    def _initialize_weights(self):
        """Set initial weights using Kaiming initialization"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        """Forward pass through the network"""
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = self.classifier(x)
        return x

# Create the model and move it to GPU if available
model = OptimizedCNN(num_classes=5).to(DEVICE)

# Count the parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

param_count = count_parameters(model)
print(f"Model parameters: {param_count:,}")
print(f"✅ Parameters < 95,000: {param_count < 95000}")
print(f"\nModel architecture:\n{model}")

Model parameters: 53,485
✅ Parameters < 95,000: True

Model architecture:
OptimizedCNN(
  (features): Sequential(
    (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, str

# CELL 4: Training Function
-------------------------
This function trains our CNN model on the data.

Improvements we added:
1. Label Smoothing - helps the model not be too confident
2. CosineAnnealing scheduler - adjusts learning rate smoothly
3. Gradient scaling for faster training on GPU (mixed precision)
4. Early stopping - stops training if no improvement
5. Patience of 15 epochs - waits longer before stopping

The function tracks:
- Training loss and accuracy
- Validation loss and accuracy
- F1-macro score (better for imbalanced data)
- Saves the best model automatically

In [ ]:
def train_model_optimized(model, train_loader, val_loader, epochs=50, lr=0.001):
    # Loss function with label smoothing (helps prevent overconfidence)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Optimizer with weight decay (regularization)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)

    # Cosine annealing scheduler - smoothly decreases learning rate
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc = 0.0
    best_epoch = 0
    patience_counter = 0
    patience = 10  # Wait 15 epochs before stopping

    # Store metrics for later analysis
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'val_f1': []
    }

    # Mixed precision training (faster on GPU)
    scaler = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()

            # Use mixed precision if available
            if scaler:
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_loss = train_loss / train_total
        train_acc = train_correct / train_total

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        val_f1 = f1_score(all_labels, all_preds, average='macro')

        # Update learning rate
        scheduler.step()

        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        # Print progress
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")

        # Early stopping and model saving
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model_optimized.pth')
            print(f"  ✅ New best model saved! (Acc: {val_acc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  ⏹ Early stopping triggered after {epoch+1} epochs")
                break

    print(f"\n✅ Training complete!")
    print(f"Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch+1}")

    return history, best_val_acc

# Train the model
history, best_val_acc = train_model_optimized(
    model,
    train_loader,
    val_loader,
    epochs=100,
    lr=0.001
)

# Print final model statistics
print(f"\n📊 Final Model Statistics:")
print(f"  Parameters: {count_parameters(model):,}")
print(f"  Best Validation Accuracy: {best_val_acc:.4f}")

Epoch 1/100 [Val]: 100%|██████████| 157/157 [00:03<00:00, 46.52it/s]


Epoch 1/100
  Train Loss: 2.4470, Train Acc: 0.2368
  Val Loss: 1.5392, Val Acc: 0.3056, Val F1: 0.1759
  LR: 0.001000
  ✅ New best model saved! (Acc: 0.3056)


Epoch 2/100 [Val]: 100%|██████████| 157/157 [00:04<00:00, 35.61it/s]


Epoch 2/100
  Train Loss: 1.5490, Train Acc: 0.2944
  Val Loss: 1.4183, Val Acc: 0.4216, Val F1: 0.3622
  LR: 0.000999
  ✅ New best model saved! (Acc: 0.4216)


Epoch 3/100 [Val]: 100%|██████████| 157/157 [00:03<00:00, 40.25it/s]


...

Epoch 89/100
  Train Loss: 0.7318, Train Acc: 0.8319
  Val Loss: 0.7300, Val Acc: 0.8346, Val F1: 0.8338
  LR: 0.000030


Epoch 90/100 [Val]: 100%|██████████| 157/157 [00:03<00:00, 42.67it/s]


Epoch 90/100
  Train Loss: 0.7266, Train Acc: 0.8356
  Val Loss: 0.7288, Val Acc: 0.8352, Val F1: 0.8346
  LR: 0.000024


Epoch 91/100 [Val]: 100%|██████████| 157/157 [00:04<00:00, 36.33it/s]

Epoch 91/100
  Train Loss: 0.7295, Train Acc: 0.8334
  Val Loss: 0.7330, Val Acc: 0.8350, Val F1: 0.8344
  LR: 0.000020
  ⏹ Early stopping triggered after 91 epochs

✅ Training complete!
Best validation accuracy: 0.8368 at epoch 81

📊 Final Model Statistics:
  Parameters: 53,485
  Best Validation Accuracy: 0.8368


# CELL 5: Optional Sanity Evaluation
-----------------------------------
This cell is only a local sanity check. **It is not useful to test metrics in the GitHub form.**

The challenge form should use the validation metrics from the best validation epoch:
- `validation_accuracy`
- `validation_f1_macro`

The submission itself is evaluated later by the repository against hidden labels.


In [ ]:
def evaluate_model(model, data_loader):
    """Optional local sanity evaluation; not the challenge leaderboard evaluation."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc="Local sanity evaluation"):
            images = images.to(DEVICE)
            outputs = model(images)
            predicted = outputs.argmax(dim=1)
            all_preds.extend(predicted.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())

    local_accuracy = accuracy_score(all_labels, all_preds)
    local_macro_f1 = f1_score(all_labels, all_preds, average='macro')
    print(f"Local sanity accuracy: {local_accuracy:.4f}")
    print(f"Local sanity F1-macro: {local_macro_f1:.4f}")
    return all_preds, all_labels, local_accuracy, local_macro_f1

# Load the best validation checkpoint before generating the final predictions.
model.load_state_dict(torch.load('best_model_optimized.pth', map_location=DEVICE))
model.eval()

# IMPORTANT: values submitted in the GitHub form come from VALIDATION, not this sanity check.
best_idx = int(np.argmax(history['val_acc']))
validation_accuracy = float(history['val_acc'][best_idx])
validation_f1_macro = float(history['val_f1'][best_idx])

print(f"Validation Accuracy for GitHub form: {validation_accuracy:.4f}")
print(f"Validation F1-macro for GitHub form: {validation_f1_macro:.4f}")
print(f"Best validation epoch: {best_idx + 1}")


Validation Accuracy for GitHub form: 0.8368
Validation F1-macro for GitHub form: 0.8366
Best validation epoch: 81


# CELL 6: Generate Predictions File
--------------------------------
We create predictions.csv with the format required for submission:
- id: test_XXXXX (e.g., test_000001)
- y_pred: integer between 0 and 4 (the predicted class)

This file is needed for the challenge submission.

**CRITICAL:** The challenge hidden test IDs are exactly `test_000001` through `test_002500`. Any ID such as `test_002501` will cause evaluation failure.


In [ ]:
def generate_predictions_csv(model, test_loader, output_file='predictions.csv'):
    """
    Generate the submission file required by the Nayarit CNN Challenge.

    The hidden evaluator expects EXACTLY:
        test_000001 ... test_002500

    No other test IDs are allowed.
    """
    model.eval()
    all_preds = []

    with torch.no_grad():
        for images, _ in tqdm(test_loader, desc="Generating challenge predictions"):
            images = images.to(DEVICE)
            outputs = model(images)
            predicted = outputs.argmax(dim=1)
            all_preds.extend(predicted.cpu().numpy().tolist())

    expected_test_size = 2500

    # Hard stop if the test loader is not exactly the challenge size.
    if len(all_preds) != expected_test_size:
        raise RuntimeError(
            f"SUBMISSION BLOCKED: generated {len(all_preds)} predictions; "
            f"the hidden evaluator requires exactly {expected_test_size}."
        )

    # IDs must be exactly test_000001 ... test_002500.
    sample_ids = [f"test_{i:06d}" for i in range(1, expected_test_size + 1)]

    df = pd.DataFrame({
        "id": sample_ids,
        "y_pred": np.asarray(all_preds, dtype=np.int64)
    })

    # Strict integrity checks.
    expected_columns = ["id", "y_pred"]
    assert df.columns.tolist() == expected_columns
    assert len(df) == 2500
    assert df["id"].iloc[0] == "test_000001"
    assert df["id"].iloc[-1] == "test_002500"
    assert df["id"].is_unique
    assert df["y_pred"].notna().all()
    assert pd.api.types.is_integer_dtype(df["y_pred"]), "y_pred must have integer dtype"
    assert df["y_pred"].between(0, 4).all()

    df.to_csv(output_file, index=False)

    print(f"✅ Predictions saved: {output_file}")
    print(f"Rows: {len(df)}")
    print(f"First ID: {df['id'].iloc[0]}")
    print(f"Last ID:  {df['id'].iloc[-1]}")
    print(f"Columns:  {df.columns.tolist()}")
    print(f"Prediction values: {sorted(df['y_pred'].unique().tolist())}")

    return df


# IMPORTANT:
# The training function saves the best checkpoint as best_model_optimized.pth.
# Always use that checkpoint for the submission instead of the final epoch.
best_checkpoint = "best_model_optimized.pth"

if os.path.exists(best_checkpoint):
    model.load_state_dict(torch.load(best_checkpoint, map_location=DEVICE))
    model.to(DEVICE)
    print(f"✅ Loaded best checkpoint: {best_checkpoint}")
else:
    print("⚠️ Best checkpoint not found; using the current model.")

predictions_df = generate_predictions_csv(
    model,
    test_loader,
    output_file="predictions.csv"
)


✅ Loaded best checkpoint: best_model_optimized.pth


Generating challenge predictions: 100%|██████████| 79/79 [00:01<00:00, 40.74it/s]


✅ Predictions saved: predictions.csv
Rows: 2500
First ID: test_000001
Last ID:  test_002500
Columns:  ['id', 'y_pred']
Prediction values: [0, 1, 2, 3, 4]


# CELL 7: Create Submission.zip
-----------------------------
This cell creates the submission package containing:
1. predictions.csv - The prediction file
2. ABLATIONS.md - Description of our experiments

The ABLATIONS.md file includes:
- Student information
- Model details (name, parameters)
- Performance metrics
- Description of experiments we tried

**CRITICAL:** The challenge hidden test IDs are exactly `test_000001` through `test_002500`. Any ID such as `test_002501` will cause evaluation failure.


In [ ]:
import os
import zipfile

def create_ablation_md(student_name="Ricardo_Torres_Lopez",
                       student_id="RicardoTL75",
                       model_name="CNN_Challenge",
                       num_params=None,
                       val_acc=None,
                       val_f1=None):
    """Create a submission-safe ABLATIONS.md without unsupported experimental claims."""

    if num_params is None:
        num_params = count_parameters(model)

    if val_acc is None:
        val_acc = best_val_acc

    # The best validation accuracy was obtained at epoch 81.
    # At that same epoch, the notebook records F1 = 0.8366.
    if val_f1 is None:
        best_acc_index = int(np.argmax(history["val_acc"]))
        val_f1 = history["val_f1"][best_acc_index]

    ablation_content = f"""# Ablation / Experiment Report

## Student Information
- **Student Name**: {student_name}
- **Student ID**: {student_id}

## Final Model
- **Model Name**: {model_name}
- **Number of Parameters**: {num_params:,}
- **Parameter Constraint (<95,000)**: PASS
- **Best Validation Accuracy**: {val_acc:.4f}
- **Validation F1-macro at Best Accuracy**: {val_f1:.4f}
- **Best Validation Epoch**: {int(np.argmax(history["val_acc"])) + 1}

## Implemented Experiments / Design Changes

### Experiment 1 — Data Augmentation
The training pipeline uses:
- RandomHorizontalFlip
- RandomRotation(10)
- ColorJitter for brightness and contrast

Validation data uses deterministic normalization without random augmentation.

### Experiment 2 — Regularization
The final CNN uses:
- Dropout(0.4)
- Dropout(0.3)
- Batch Normalization
- AdamW with weight decay = 5e-4
- CrossEntropyLoss with label smoothing = 0.1

### Experiment 3 — Learning-Rate Schedule
The training configuration uses CosineAnnealingLR to progressively reduce the learning rate during training.

### Experiment 4 — Early Stopping and Best-Checkpoint Selection
Training uses early stopping. The checkpoint with the best validation accuracy is saved as:
`best_model_optimized.pth`

For the competition submission, that best checkpoint is loaded before generating predictions.

## Final Architecture
- 3 convolutional blocks
- Convolution channels: 8, 16, 32
- Batch Normalization
- ReLU activation
- Max Pooling
- Fully connected layers: 64 -> 32 -> 5
- Dropout: 0.4 and 0.3
- Kaiming initialization

## Results
The notebook records:
- **Parameters**: {num_params:,}
- **Best Validation Accuracy**: {val_acc:.4f}
- **Validation F1-macro at the same best-accuracy epoch**: {val_f1:.4f}

## Important Note
This report describes the experiments and techniques actually implemented in the notebook.
No unsupported numeric improvement claims are included for individual ablations.

## Submission Files
The ZIP must contain exactly:
- `predictions.csv`
- `ABLATIONS.md`
"""

    with open("ABLATIONS.md", "w", encoding="utf-8") as f:
        f.write(ablation_content)

    print("✅ ABLATIONS.md created")


def create_submission_zip():
    """Create submission.zip with exactly the two required files."""
    required = ["predictions.csv", "ABLATIONS.md"]

    for filename in required:
        if not os.path.exists(filename):
            raise FileNotFoundError(f"Missing required file: {filename}")

    # Remove an old ZIP so stale files cannot remain inside it.
    if os.path.exists("submission.zip"):
        os.remove("submission.zip")

    with zipfile.ZipFile("submission.zip", "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        for filename in required:
            zipf.write(filename, arcname=filename)

    with zipfile.ZipFile("submission.zip", "r") as zipf:
        names = zipf.namelist()

    assert names == required, f"Unexpected ZIP contents: {names}"

    print("✅ submission.zip created")
    print(f"Contents: {names}")


# Use the best validation epoch's metrics.
best_idx = int(np.argmax(history["val_acc"]))
submission_val_acc = history["val_acc"][best_idx]
submission_val_f1 = history["val_f1"][best_idx]

create_ablation_md(
    num_params=count_parameters(model),
    val_acc=submission_val_acc,
    val_f1=submission_val_f1
)
create_submission_zip()


✅ ABLATIONS.md created
✅ submission.zip created
Contents: ['predictions.csv', 'ABLATIONS.md']


# CELL 8: Final Validation and Summary
------------------------------------
This cell checks all files and displays a summary report.
It confirms everything is ready for submission.

In [ ]:
import zipfile
import os
import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL CHALLENGE SUBMISSION VALIDATION")
print("=" * 70)

# Required files
required_files = ["predictions.csv", "ABLATIONS.md", "submission.zip"]
for filename in required_files:
    assert os.path.exists(filename), f"Missing file: {filename}"
    print(f"✅ Exists: {filename}")

# Validate predictions.csv
df = pd.read_csv("predictions.csv")

expected_ids = [f"test_{i:06d}" for i in range(1, 2501)]
expected_columns = ["id", "y_pred"]

assert df.columns.tolist() == expected_columns, (
    f"Columns must be exactly {expected_columns}, got {df.columns.tolist()}"
)
assert len(df) == 2500, f"Expected 2500 rows, got {len(df)}"
assert df["id"].tolist() == expected_ids, (
    "ID sequence does not exactly match test_000001 ... test_002500"
)
assert df["id"].is_unique, "Duplicate test IDs found"
assert df["y_pred"].notna().all(), "Missing predictions found"
assert df["y_pred"].apply(lambda x: int(x) in range(5)).all(), (
    "Predictions must be integers 0,1,2,3,4"
)

print("\n📊 predictions.csv")
print(f"   Rows: {len(df)}")
print(f"   Columns: {df.columns.tolist()}")
print(f"   First ID: {df['id'].iloc[0]}")
print(f"   Last ID:  {df['id'].iloc[-1]}")
print(f"   Unique IDs: {df['id'].nunique()}")
print(f"   Prediction classes: {sorted(df['y_pred'].unique().tolist())}")

# Validate ZIP contents
with zipfile.ZipFile("submission.zip", "r") as zipf:
    zip_names = zipf.namelist()

assert zip_names == ["predictions.csv", "ABLATIONS.md"], (
    f"ZIP must contain exactly predictions.csv and ABLATIONS.md; got {zip_names}"
)

print("\n📦 submission.zip")
print(f"   Contents: {zip_names}")

# Validate ABLATIONS.md
with open("ABLATIONS.md", "r", encoding="utf-8") as f:
    abl = f.read()

assert "Ablation" in abl
assert "Experiment" in abl
print(f"   ABLATIONS.md size: {len(abl)} characters")

# Model / metric summary
param_count = count_parameters(model)
best_idx = int(np.argmax(history["val_acc"]))
val_acc = history["val_acc"][best_idx]
val_f1 = history["val_f1"][best_idx]

assert param_count < 95000

print("\n📈 SUBMISSION METRICS")
print(f"   Parameters: {param_count:,}")
print(f"   Validation Accuracy: {val_acc:.4f}")
print(f"   Validation F1-macro: {val_f1:.4f}")
print(f"   Best Epoch: {best_idx + 1}")

print("\n" + "=" * 70)
print("🎉 ALL LOCAL SUBMISSION CHECKS PASSED")
print("🎯 The prediction IDs are exactly test_000001 ... test_002500")
print("=" * 70)


FINAL CHALLENGE SUBMISSION VALIDATION
✅ Exists: predictions.csv
✅ Exists: ABLATIONS.md
✅ Exists: submission.zip

📊 predictions.csv
   Rows: 2500
   Columns: ['id', 'y_pred']
   First ID: test_000001
   Last ID:  test_002500
   Unique IDs: 2500
   Prediction classes: [0, 1, 2, 3, 4]

📦 submission.zip
   Contents: ['predictions.csv', 'ABLATIONS.md']
   ABLATIONS.md size: 1934 characters

📈 SUBMISSION METRICS
   Parameters: 53,485
   Validation Accuracy: 0.8368
   Validation F1-macro: 0.8366
   Best Epoch: 81

🎉 ALL LOCAL SUBMISSION CHECKS PASSED
🎯 The prediction IDs are exactly test_000001 ... test_002500


# CELL 9: Download the Submission ZIP
Run this last cell only after **all previous cells finish successfully**. It verifies the ZIP size/content and downloads the exact `submission.zip` that must be attached to the GitHub issue form.


In [ ]:
# CELL 9: Download the exact ZIP to upload to GitHub
# --------------------------------------------------
# GitHub's issue form accepts .zip files up to 25 MB.
zip_path = os.path.abspath("submission.zip")
zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)

assert zip_size_mb < 25, f"submission.zip is too large: {zip_size_mb:.2f} MB"

with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    assert names == ["predictions.csv", "ABLATIONS.md"]
    assert all("/" not in name and "\\" not in name for name in names), (
        "Required files must be at the root of the ZIP"
    )

print(f"READY TO UPLOAD: {zip_path}")
print(f"ZIP size: {zip_size_mb:.3f} MB")
print("ZIP root contents:", names)

# In Google Colab this triggers the browser download.
try:
    from google.colab import files
    files.download("submission.zip")
except ImportError:
    print("Not running in Colab. Download submission.zip from the notebook file browser.")

READY TO UPLOAD: /content/submission.zip
ZIP size: 0.007 MB
ZIP root contents: ['predictions.csv', 'ABLATIONS.md']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>